# Blood Cell Segmentation Pipeline

### Task 1: Computation

Given:
- Input size = 256 × 256
- After 2 pooling layers (2×2, stride 2):
  - Pooling 1: 128 × 128
  - Pooling 2: 64 × 64
  - **Output size after pooling = 64 × 64**
- After 2 upsampling layers (2×2):
  - Upsampling 1: 128 × 128
  - Upsampling 2: 256 × 256
  - **Output size after upsampling = 256 × 256**

In [ ]:
%%bash
if [ -f "archive.zip" ]; then
    unzip -q -o archive.zip -d ./blood_cells_dataset
elif [ -d "./blood_cells_dataset" ]; then
    echo "Dataset already exists."
else
    kaggle datasets download -d paultimothymooney/blood-cells --force
    unzip -q blood-cells.zip -d ./blood_cells_dataset
fi

In [ ]:
import os
import zipfile
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
image_paths = glob.glob("./blood_cells_dataset/**/*.jpeg", recursive=True) + glob.glob("./blood_cells_dataset/**/*.png", recursive=True)
if not image_paths and os.path.exists("archive.zip"):
    with zipfile.ZipFile("archive.zip", "r") as zip_ref:
        zip_ref.extractall("./blood_cells_dataset")
    image_paths = glob.glob("./blood_cells_dataset/**/*.jpeg", recursive=True) + glob.glob("./blood_cells_dataset/**/*.png", recursive=True)

img_path = image_paths[1]
img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
preprocessed_image = cv2.resize(gray, (256, 256))
_, segmented_output = cv2.threshold(preprocessed_image, 127, 255, cv2.THRESH_BINARY)

In [ ]:
inputs = layers.Input(shape=(256, 256, 1))
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
outputs = layers.Conv2D(1, (1, 1), activation="sigmoid", padding="same")(x)
model = models.Model(inputs, outputs)
model.summary()

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Preprocessed Image")
plt.imshow(preprocessed_image, cmap="gray")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("Segmented Output")
plt.imshow(segmented_output, cmap="gray")
plt.axis("off")
plt.show()